# 13b District-Period Panel

Leakage-free future-impact ranking workflow. Existing raw inputs are read but not modified.

In [1]:

from pathlib import Path
import hashlib, re, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats, optimize

pd.set_option('display.max_columns', 180)
pd.set_option('display.max_rows', 160)
plt.style.use('seaborn-v0_8-whitegrid')
PROJECT_ROOT = Path.cwd() if (Path.cwd() / 'data').exists() else Path.cwd().parent
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
TABLE_DIR = PROJECT_ROOT / 'outputs' / 'tables'
FIGURE_DIR = PROJECT_ROOT / 'outputs' / 'figures'
REPORT_DIR = PROJECT_ROOT / 'outputs' / 'reports'
PROC_DIR = PROJECT_ROOT / 'data' / 'processed'
for d in [TABLE_DIR, FIGURE_DIR, REPORT_DIR, PROC_DIR]: d.mkdir(parents=True, exist_ok=True)
RANDOM_SEED=42
BOOTSTRAPS=2000
PERIODS=[('1967-1985',1967,1985),('1986-2004',1986,2004),('2005-2023',2005,2023)]
TRANSITIONS=[('historical_replication','1967-1985','1986-2004'),('primary_temporal_test','1986-2004','2005-2023')]

def sha256_file(path):
    h=hashlib.sha256()
    with path.open('rb') as f:
        for block in iter(lambda:f.read(1024*1024), b''): h.update(block)
    return h.hexdigest()

def norm_text(x):
    if pd.isna(x): return ''
    s=str(x).strip().lower()
    s=s.replace('&',' and ')
    s=re.sub(r'[^a-z0-9]+',' ',s)
    return re.sub(r'\s+',' ',s).strip()

def split_list_cell(x):
    if pd.isna(x): return []
    s=str(x).strip()
    if not s or s.lower() in {'nan','none'}: return []
    return [p.strip() for p in s.split(',') if p.strip() and p.strip().lower() not in {'nan'}]

def numeric_series(s):
    return pd.to_numeric(s, errors='coerce').astype('float64')

def parse_code(x):
    if pd.isna(x): return ''
    s=str(x).strip()
    if not s or s.lower() in {'nan','none'}: return ''
    if re.fullmatch(r'\d+\.0', s): s=s[:-2]
    return s

def safe_spearman(x,y):
    dat=pd.concat([pd.Series(x),pd.Series(y)],axis=1).replace([np.inf,-np.inf],np.nan).dropna()
    if len(dat)<4 or dat.iloc[:,0].nunique()<2 or dat.iloc[:,1].nunique()<2: return np.nan
    return float(stats.spearmanr(dat.iloc[:,0], dat.iloc[:,1]).statistic)

def safe_kendall(x,y):
    dat=pd.concat([pd.Series(x),pd.Series(y)],axis=1).replace([np.inf,-np.inf],np.nan).dropna()
    if len(dat)<4 or dat.iloc[:,0].nunique()<2 or dat.iloc[:,1].nunique()<2: return np.nan
    return float(stats.kendalltau(dat.iloc[:,0], dat.iloc[:,1]).statistic)

RAW_HASHES_BEFORE={p.name: sha256_file(p) for p in sorted(RAW_DIR.glob('*.csv'))}
BASELINE_PATH=PROC_DIR/'dfsi_verified_baseline.csv'
long=pd.read_csv(TABLE_DIR/'predictive_ranking_event_district_long.csv')
baseline=pd.read_csv(BASELINE_PATH)
feas=pd.read_csv(TABLE_DIR/'predictive_ranking_feasibility_status.csv')
if feas.loc[feas['criterion'].eq('overall_feasibility_status'),'notes'].iloc[0] != 'PROCEED':
    raise RuntimeError('Feasibility gate did not pass; district-period panel not constructed.')
# Preserve the original verified-baseline row index because the feasibility matching table used it as baseline_district_id.
universe=baseline[baseline['join_status'].eq('one_to_one_comparison_record')].copy()
universe['baseline_district_id']=universe.index.astype(str)
universe=universe.reset_index(drop=True)
# Ensure population is only secondary sensitivity.
universe['population']=numeric_series(universe['population'])
long['year']=numeric_series(long['year'])
long['duration_days']=numeric_series(long['duration_days'])
long['fatalities_for_district_primary']=numeric_series(long['fatalities_for_district_primary']).fillna(0)
long['injuries_for_district_primary']=numeric_series(long['injuries_for_district_primary']).fillna(0)
long['fatalities_reported']=numeric_series(long['fatalities_reported'])
long['injuries_reported']=numeric_series(long['injuries_reported'])
long['baseline_district_id_norm']=long['baseline_district_id'].apply(parse_code)
matched=long[long['match_status'].eq('matched')].copy()
rows=[]
for _,u in universe.iterrows():
    did=str(u['baseline_district_id'])
    for label,start,end in PERIODS:
        p=matched[(matched['baseline_district_id_norm']==did)&(matched['year'].between(start,end))].copy()
        event_count=len(p)
        duration_nonmiss=p['duration_days'].dropna()
        impact=p[p['district_specific_impact'].astype(str).str.lower().isin(['true','1'])] if 'district_specific_impact' in p.columns else p.iloc[0:0]
        fatal_reported_events=int(impact['fatalities_reported'].notna().sum()) if len(impact) else 0
        injury_reported_events=int(impact['injuries_reported'].notna().sum()) if len(impact) else 0
        fatal_events=int((impact['fatalities_for_district_primary']>0).sum()) if len(impact) else 0
        injury_events=int((impact['injuries_for_district_primary']>0).sum()) if len(impact) else 0
        distinct_years=int(p['year'].dropna().nunique()) if event_count else 0
        last_year=p['year'].dropna().max() if event_count else np.nan
        rows.append({'baseline_district_id':did,'State_Name':u['State_Name'],'district_name':u['district_name'],'period':label,'start_year':start,'end_year':end,'population_secondary':u['population'],'event_count':event_count,'total_flood_duration':float(duration_nonmiss.sum()) if len(duration_nonmiss) else 0.0,'mean_flood_duration':float(duration_nonmiss.mean()) if len(duration_nonmiss) else 0.0,'maximum_event_duration':float(duration_nonmiss.max()) if len(duration_nonmiss) else 0.0,'fatality_count':float(impact['fatalities_for_district_primary'].sum()) if len(impact) else 0.0,'injury_count':float(impact['injuries_for_district_primary'].sum()) if len(impact) else 0.0,'fatal_event_count':fatal_events,'injury_event_count':injury_events,'fatality_reported_events':fatal_reported_events,'injury_reported_events':injury_reported_events,'proportion_events_with_fatalities':fatal_events/fatal_reported_events if fatal_reported_events else np.nan,'proportion_events_with_injuries':injury_events/injury_reported_events if injury_reported_events else np.nan,'years_since_recent_flood':(end-last_year) if pd.notna(last_year) else (end-start+1),'distinct_flood_years':distinct_years,'average_annual_event_count':event_count/(end-start+1),'average_annual_fatalities':(float(impact['fatalities_for_district_primary'].sum()) if len(impact) else 0.0)/(end-start+1),'average_annual_injuries':(float(impact['injuries_for_district_primary'].sum()) if len(impact) else 0.0)/(end-start+1),'zero_event_status':'credible_zero_no_matched_events' if event_count==0 else 'matched_events_present','impact_assignment_note':'Fatalities/injuries use only district-specific or single-district event records; multi-district impact totals not allocated.'})
panel=pd.DataFrame(rows)
panel.to_csv(TABLE_DIR/'predictive_ranking_district_period_panel.csv',index=False)
comp=[]
for label,_,_ in PERIODS:
    p=panel[panel['period']==label]
    comp.append({'period':label,'districts':len(p),'zero_event_districts':int((p['event_count']==0).sum()),'districts_positive_fatalities':int((p['fatality_count']>0).sum()),'total_primary_fatalities':float(p['fatality_count'].sum()),'median_fatalities':float(p['fatality_count'].median()),'fatality_zero_percent':float((p['fatality_count']==0).mean()*100),'districts_positive_injuries':int((p['injury_count']>0).sum()),'total_primary_injuries':float(p['injury_count'].sum()),'missing_population_secondary':int(p['population_secondary'].isna().sum()),'notes':'Zero-event districts preserved from verified one-to-one baseline universe; impacts only district-assignable.'})
pd.DataFrame(comp).to_csv(TABLE_DIR/'predictive_ranking_panel_completeness.csv',index=False)
display(pd.DataFrame(comp))
RAW_HASHES_AFTER={p.name: sha256_file(p) for p in sorted(RAW_DIR.glob('*.csv'))}
assert RAW_HASHES_BEFORE==RAW_HASHES_AFTER, 'data/raw changed during panel construction.'


,period,districts,zero_event_districts,districts_positive_fatalities,total_primary_fatalities,median_fatalities,fatality_zero_percent,districts_positive_injuries,total_primary_injuries,missing_population_secondary,notes
0,1967-1985,720,170,99,3201.0,0.0,86.250000,34,587.0,0,Zero-event districts preserved from verified o...
1,1986-2004,720,171,162,3202.0,0.0,77.500000,90,1340.0,0,Zero-event districts preserved from verified o...
2,2005-2023,720,152,332,3654.0,0.0,53.888889,137,1096.0,0,Zero-event districts preserved from verified o...
